# Velociraptor Training Notebook

Train a velociraptor to walk, run, and strike prey using reinforcement learning.

**Training Stages:**
1. **Balance** - Learn to stand without falling
2. **Locomotion** - Walk and run forward
3. **Strike** - Sprint and attack prey with sickle claws

This notebook uses MuJoCo + Gymnasium + Stable-Baselines3 (PPO).

## 1. Setup & Installation

In [ ]:
# Install dependencies (uncomment for Colab)
# !pip install mujoco>=3.0.0 gymnasium>=0.29.0 stable-baselines3[extra]>=2.2.0 mediapy matplotlib

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Add repo root to path
repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import gymnasium as gym
import mujoco

print(f"MuJoCo version: {mujoco.__version__}")
print(f"Gymnasium version: {gym.__version__}")
print(f"Repo root: {repo_root}")

## 2. Load and Explore the Environment

In [ ]:
from environments.velociraptor.envs.raptor_env import RaptorEnv

# Create environment
env = RaptorEnv()

print("Environment loaded successfully!")
print(f"\nObservation space: {env.observation_space}")
print(f"  Shape: {env.observation_space.shape}")
print(f"\nAction space: {env.action_space}")
print(f"  Shape: {env.action_space.shape}")
print(f"  Range: [{env.action_space.low[0]}, {env.action_space.high[0]}]")

In [ ]:
# Print model information
model = env.model
print("Model Information:")
print(f"  Bodies: {model.nbody}")
print(f"  Joints: {model.njnt}")
print(f"  Actuators: {model.nu}")
print(f"  Sensors: {model.nsensor}")
print(f"  Total DOF: {model.nv}")
print(f"  Total mass: {sum(model.body_mass):.2f} kg")
print(f"  Timestep: {model.opt.timestep * 1000:.1f} ms")

print("\nActuators:")
for i in range(model.nu):
    name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, i)
    print(f"  [{i:2d}] {name}")

## 3. Test Environment with Random Actions

In [ ]:
def run_random_episode(env, max_steps=200, seed=42):
    """Run a single episode with random actions."""
    obs, info = env.reset(seed=seed)

    total_reward = 0
    rewards = []

    for step in range(max_steps):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)

        total_reward += reward
        rewards.append(reward)

        if terminated or truncated:
            reason = info.get("termination_reason", "truncated")
            print(f"Episode ended at step {step + 1}: {reason}")
            break

    return total_reward, rewards, step + 1


# Run multiple random episodes
n_episodes = 5
episode_rewards = []
episode_lengths = []

for ep in range(n_episodes):
    total_reward, rewards, length = run_random_episode(env, seed=ep)
    episode_rewards.append(total_reward)
    episode_lengths.append(length)
    print(f"  Episode {ep + 1}: reward={total_reward:.2f}, length={length}")

print("\nRandom policy baseline:")
print(f"  Avg reward: {np.mean(episode_rewards):.2f} +/- {np.std(episode_rewards):.2f}")
print(f"  Avg length: {np.mean(episode_lengths):.1f} +/- {np.std(episode_lengths):.1f}")

## 4. Curriculum Learning Configuration

Training proceeds in three stages with different reward weights.

In [ ]:
# Curriculum stage configurations
STAGE_CONFIGS = {
    1: {
        "name": "balance",
        "description": "Learn to stand and balance without falling",
        "env_kwargs": {
            "forward_vel_weight": 0.0,  # No forward reward
            "alive_bonus": 1.0,  # Strong alive bonus
            "energy_penalty_weight": 0.0005,
            "tail_stability_weight": 0.1,
            "strike_bonus": 0.0,
            "strike_approach_weight": 0.0,
            "prey_distance_range": (10.0, 15.0),
            "max_episode_steps": 500,
        },
        "ppo_kwargs": {
            "learning_rate": 3e-4,
            "n_steps": 2048,
            "batch_size": 64,
            "n_epochs": 10,
            "gamma": 0.99,
            "gae_lambda": 0.95,
            "clip_range": 0.2,
            "ent_coef": 0.01,
        },
        "timesteps": 500_000,
    },
    2: {
        "name": "locomotion",
        "description": "Learn forward walking/running",
        "env_kwargs": {
            "forward_vel_weight": 1.0,
            "alive_bonus": 0.5,
            "energy_penalty_weight": 0.001,
            "tail_stability_weight": 0.05,
            "strike_bonus": 0.0,
            "strike_approach_weight": 0.2,
            "prey_distance_range": (8.0, 12.0),
            "max_episode_steps": 1000,
        },
        "ppo_kwargs": {
            "learning_rate": 1e-4,
            "n_steps": 2048,
            "batch_size": 128,
            "n_epochs": 10,
            "gamma": 0.99,
            "gae_lambda": 0.95,
            "clip_range": 0.2,
            "ent_coef": 0.005,
        },
        "timesteps": 1_000_000,
    },
    3: {
        "name": "strike",
        "description": "Sprint and strike prey with sickle claw",
        "env_kwargs": {
            "forward_vel_weight": 1.0,
            "alive_bonus": 0.1,
            "energy_penalty_weight": 0.001,
            "tail_stability_weight": 0.02,
            "strike_bonus": 500.0,
            "strike_approach_weight": 0.5,
            "prey_distance_range": (3.0, 8.0),
            "prey_lateral_range": (-1.5, 1.5),
            "max_episode_steps": 1000,
        },
        "ppo_kwargs": {
            "learning_rate": 5e-5,
            "n_steps": 4096,
            "batch_size": 256,
            "n_epochs": 10,
            "gamma": 0.995,
            "gae_lambda": 0.95,
            "clip_range": 0.1,
            "ent_coef": 0.001,
        },
        "timesteps": 2_000_000,
    },
}

for stage, config in STAGE_CONFIGS.items():
    print(f"Stage {stage}: {config['name']}")
    print(f"  {config['description']}")
    print(f"  Timesteps: {config['timesteps']:,}")
    print()

## 5. Training Setup

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CallbackList, CheckpointCallback, EvalCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize


def make_env(stage, rank, seed=0):
    """Create a single environment instance."""

    def _init():
        env_kwargs = STAGE_CONFIGS[stage]["env_kwargs"].copy()
        env = RaptorEnv(**env_kwargs)
        env = Monitor(env)
        env.reset(seed=seed + rank)
        return env

    set_random_seed(seed)
    return _init


def create_vec_env(stage, n_envs=4, seed=0):
    """Create vectorized environment with normalization."""
    env = DummyVecEnv([make_env(stage, i, seed) for i in range(n_envs)])
    env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.0, clip_reward=10.0)
    return env


print("Training utilities loaded.")

## 6. Train Stage 1: Balance

The raptor learns to stand without falling over.

In [ ]:
# Configuration
STAGE = 1
N_ENVS = 4
SEED = 42

# For quick testing, use fewer timesteps
QUICK_TEST = True
TIMESTEPS = 50_000 if QUICK_TEST else STAGE_CONFIGS[STAGE]["timesteps"]

print(f"Training Stage {STAGE}: {STAGE_CONFIGS[STAGE]['name']}")
print(f"Timesteps: {TIMESTEPS:,}")
print(f"Parallel envs: {N_ENVS}")

In [ ]:
# Create environments
train_env = create_vec_env(STAGE, N_ENVS, SEED)
eval_env = create_vec_env(STAGE, 1, SEED + 1000)

# Create model
config = STAGE_CONFIGS[STAGE]
ppo_kwargs = config["ppo_kwargs"].copy()
ppo_kwargs["verbose"] = 1

model = PPO("MlpPolicy", train_env, **ppo_kwargs)

print("\nModel created:")
print("  Policy: MlpPolicy")
print(f"  Learning rate: {ppo_kwargs['learning_rate']}")
print(f"  Batch size: {ppo_kwargs['batch_size']}")

In [ ]:
# Setup logging
log_dir = Path(f"../logs/stage{STAGE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
log_dir.mkdir(parents=True, exist_ok=True)
model_dir = log_dir / "models"
model_dir.mkdir(exist_ok=True)

# Callbacks
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=str(model_dir),
    log_path=str(log_dir),
    eval_freq=max(5000 // N_ENVS, 1),
    n_eval_episodes=3,
    deterministic=True,
)

checkpoint_callback = CheckpointCallback(
    save_freq=max(10000 // N_ENVS, 1),
    save_path=str(model_dir),
    name_prefix=f"stage{STAGE}",
)

callbacks = CallbackList([eval_callback, checkpoint_callback])

print(f"Logging to: {log_dir}")

In [ ]:
# Train!
print(f"\nStarting training for {TIMESTEPS:,} timesteps...")
print("=" * 60)

model.learn(
    total_timesteps=TIMESTEPS,
    callback=callbacks,
    progress_bar=True,
)

print("=" * 60)
print("Training complete!")

In [ ]:
# Save final model
final_path = model_dir / f"stage{STAGE}_final"
model.save(str(final_path))
train_env.save(str(final_path) + "_vecnorm.pkl")

print(f"Model saved to: {final_path}.zip")
print(f"VecNormalize saved to: {final_path}_vecnorm.pkl")

## 7. Evaluate Trained Policy

In [ ]:
def evaluate_policy(model, env, n_episodes=5):
    """Evaluate a trained policy."""
    episode_rewards = []
    episode_lengths = []

    for ep in range(n_episodes):
        obs, _ = env.reset(seed=ep + 100)
        total_reward = 0
        step = 0

        while True:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            step += 1

            if terminated or truncated:
                break

        episode_rewards.append(total_reward)
        episode_lengths.append(step)
        print(f"  Episode {ep + 1}: reward={total_reward:.2f}, length={step}")

    return episode_rewards, episode_lengths


# Evaluate
eval_env_single = RaptorEnv(**STAGE_CONFIGS[STAGE]["env_kwargs"])
print(f"\nEvaluating trained policy (Stage {STAGE})...")
rewards, lengths = evaluate_policy(model, eval_env_single, n_episodes=5)

print("\nResults:")
print(f"  Mean reward: {np.mean(rewards):.2f} +/- {np.std(rewards):.2f}")
print(f"  Mean length: {np.mean(lengths):.1f} +/- {np.std(lengths):.1f}")

eval_env_single.close()

## 8. Visualize Training Progress

In [ ]:
# Plot training curve from evaluations
eval_log = log_dir / "evaluations.npz"

if eval_log.exists():
    data = np.load(eval_log)
    timesteps = data["timesteps"]
    results = data["results"]

    mean_rewards = np.mean(results, axis=1)
    std_rewards = np.std(results, axis=1)

    plt.figure(figsize=(10, 5))
    plt.plot(timesteps, mean_rewards, "b-", label="Mean Reward")
    plt.fill_between(timesteps, mean_rewards - std_rewards, mean_rewards + std_rewards, alpha=0.3)
    plt.xlabel("Timesteps")
    plt.ylabel("Reward")
    plt.title(f"Stage {STAGE} Training Progress")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("No evaluation log found yet.")

## 9. Continue to Next Stage

To continue training with the next curriculum stage, load the saved model and create a new environment with updated reward weights.

In [ ]:
# Example: Load Stage 1 model for Stage 2 training
# Uncomment and modify the path as needed

# NEXT_STAGE = 2
# model_path = "path/to/stage1_final.zip"
#
# train_env_2 = create_vec_env(NEXT_STAGE, N_ENVS, SEED)
# model_2 = PPO.load(model_path, env=train_env_2)
#
# # Update hyperparameters for new stage
# config_2 = STAGE_CONFIGS[NEXT_STAGE]
# model_2.learning_rate = config_2["ppo_kwargs"]["learning_rate"]
# model_2.ent_coef = config_2["ppo_kwargs"]["ent_coef"]
#
# # Continue training...
# model_2.learn(total_timesteps=config_2["timesteps"], progress_bar=True)

print("See Stage 6 for training. Modify STAGE variable and re-run cells.")

## 10. Cleanup

In [ ]:
# Close environments
train_env.close()
eval_env.close()
env.close()

print("Environments closed.")